# 🫀 Heart Disease Prediction — Multiple Model Comparison System

This notebook trains and rigorously compares **5 machine learning models** on the Heart Disease dataset.  
Each model is evaluated on Accuracy, Precision, Recall, F1-Score, ROC-AUC, 5-Fold Cross-Validation, and Training Time.  
A final ranked summary table and visualisations are produced, followed by a plain-text recommendation.

**Dataset:** `heart.csv`  
**Target column:** `HeartDisease` (binary: 0 = No Disease, 1 = Disease)  
**Rows × Columns:** 918 × 12

---
## 1. Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)

# ── Reproducibility ──────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Plotting style ───────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

print('All libraries imported successfully ✅')

---
## 2. Data Loading & Exploration

In [ ]:
# ── Load dataset ─────────────────────────────────────────────────────────────
# Change the path below if your file is located elsewhere
DATA_PATH = 'heart.csv'

df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# ── Basic info ────────────────────────────────────────────────────────────────
df.info()

In [ ]:
# ── Statistical summary ───────────────────────────────────────────────────────
df.describe(include='all').T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

In [ ]:
# ── Missing values ────────────────────────────────────────────────────────────
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing >= 0].to_string())
print(f'\nTotal missing cells: {missing.sum()}')

In [ ]:
# ── Target distribution ───────────────────────────────────────────────────────
TARGET = 'HeartDisease'
counts = df[TARGET].value_counts()
labels = ['No Disease (0)', 'Heart Disease (1)']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(labels, counts.values, color=['#5b9bd5', '#ed7d31'], edgecolor='white', width=0.5)
axes[0].set_title('Target Class Distribution (Count)', fontweight='bold')
axes[0].set_ylabel('Number of Samples')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=labels, autopct='%1.1f%%',
            colors=['#5b9bd5', '#ed7d31'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Target Class Distribution (%)', fontweight='bold')

plt.suptitle('HeartDisease — Class Balance', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature distributions ─────────────────────────────────────────────────────
num_cols = df.select_dtypes(include=np.number).columns.drop(TARGET).tolist()
# Robust detection: works with pandas object, string, and StringDtype
cat_cols = [c for c in df.columns if c != TARGET and
            (str(df[c].dtype) in ('object', 'string', 'category')
             or 'string' in str(df[c].dtype).lower()
             or type(df[c].dtype).__name__ == 'StringDtype')]

print(f'Numerical features  ({len(num_cols)}): {num_cols}')
print(f'Categorical features ({len(cat_cols)}): {cat_cols}')

# Histograms for numeric columns
n = len(num_cols)
cols_per_row = 3
rows = (n + cols_per_row - 1) // cols_per_row

fig, axes = plt.subplots(rows, cols_per_row, figsize=(14, rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for cls, color, lbl in zip([0, 1], ['#5b9bd5', '#ed7d31'], ['No Disease', 'Disease']):
        axes[i].hist(df[df[TARGET] == cls][col], bins=25, alpha=0.6,
                     color=color, label=lbl, edgecolor='white')
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')
    axes[i].legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Feature Distributions by Target Class',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Categorical feature counts ────────────────────────────────────────────────
n_cat = len(cat_cols)
fig, axes = plt.subplots(1, n_cat, figsize=(4 * n_cat, 4))
if n_cat == 1:
    axes = [axes]

for ax, col in zip(axes, cat_cols):
    ct = pd.crosstab(df[col], df[TARGET])
    ct.plot(kind='bar', ax=ax, color=['#5b9bd5', '#ed7d31'],
            edgecolor='white', rot=0, legend=True)
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Count')
    ax.legend(['No Disease', 'Disease'], fontsize=8)

plt.suptitle('Categorical Features vs Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap (numeric only) ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
corr = df[num_cols + [TARGET]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, square=True, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix — Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Preprocessing Pipeline

- **Numerical features**: median imputation → standard scaling  
- **Categorical features**: most-frequent imputation → one-hot encoding  
- **Target**: `HeartDisease` (already binary integer, no transformation needed)  
- **Train / test split**: 80 % / 20 %, stratified on target

In [ ]:
# ── Feature / target split ────────────────────────────────────────────────────
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Re-identify column types from X
num_features = X.select_dtypes(include=np.number).columns.tolist()
cat_features = [c for c in X.columns
                if (str(X[c].dtype) in ('object', 'string', 'category')
                    or 'string' in str(X[c].dtype).lower()
                    or type(X[c].dtype).__name__ == 'StringDtype')]

print('Numerical features :', num_features)
print('Categorical features:', cat_features)

# ── Train / test split ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f'\nTraining samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')

# ── Column transformer (shared preprocessing) ─────────────────────────────────
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_features),
    ('cat', categorical_transformer, cat_features)
])

print('\nPreprocessor built ✅')

---
## 4. Model Definitions

Each model is wrapped in a `Pipeline` that prepends the **identical** preprocessor,  
ensuring a perfectly fair apples-to-apples comparison.

In [ ]:
from sklearn.pipeline import Pipeline as SkPipeline

def make_pipeline(classifier):
    """Wrap a classifier with the shared preprocessing pipeline."""
    return SkPipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', classifier)
    ])

models = {
    'Logistic Regression': make_pipeline(
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    ),
    'Decision Tree': make_pipeline(
        DecisionTreeClassifier(random_state=RANDOM_STATE)
    ),
    'Random Forest': make_pipeline(
        RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
    ),
    'SVM': make_pipeline(
        SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)
    ),
    'KNN': make_pipeline(
        KNeighborsClassifier(n_neighbors=7)
    ),
}

print('Models defined:')
for name in models:
    print(f'  • {name}')

---
## 5. Training & Evaluation

For each model we record:
- Accuracy, Precision, Recall, F1, ROC-AUC on the held-out test set  
- 5-Fold Stratified Cross-Validation mean ± std (on training data)  
- Wall-clock training time  
- Confusion matrix plot

In [ ]:
CV_FOLDS = 5
skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

results = []          # list of metric dicts
trained_models = {}   # store fitted pipelines for later use


def evaluate_model(name, pipeline, X_tr, y_tr, X_te, y_te):
    """
    Fit the pipeline, measure training time, compute all evaluation metrics,
    and return a results dict plus the fitted pipeline.
    """
    # ── Train ─────────────────────────────────────────────────────────────────
    t0 = time.perf_counter()
    pipeline.fit(X_tr, y_tr)
    train_time = time.perf_counter() - t0

    # ── Predict ───────────────────────────────────────────────────────────────
    y_pred = pipeline.predict(X_te)
    y_prob = pipeline.predict_proba(X_te)[:, 1]

    # ── Metrics ───────────────────────────────────────────────────────────────
    acc  = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec  = recall_score(y_te, y_pred, zero_division=0)
    f1   = f1_score(y_te, y_pred, zero_division=0)
    auc  = roc_auc_score(y_te, y_prob)

    # ── Cross-validation ──────────────────────────────────────────────────────
    cv_scores = cross_val_score(pipeline, X_tr, y_tr, cv=skf, scoring='f1', n_jobs=-1)
    cv_mean   = cv_scores.mean()
    cv_std    = cv_scores.std()

    return {
        'Model'      : name,
        'Accuracy'   : round(acc,  4),
        'Precision'  : round(prec, 4),
        'Recall'     : round(rec,  4),
        'F1-Score'   : round(f1,   4),
        'ROC-AUC'    : round(auc,  4),
        'CV Mean F1' : round(cv_mean, 4),
        'CV Std F1'  : round(cv_std,  4),
        'CV Summary' : f'{cv_mean:.4f} ± {cv_std:.4f}',
        'Train Time (s)': round(train_time, 3),
    }, pipeline


# ── Run evaluation for every model ───────────────────────────────────────────
for model_name, pipeline in models.items():
    print(f'Training: {model_name} ...', end='  ')
    metrics, fitted = evaluate_model(
        model_name, pipeline, X_train, y_train, X_test, y_test
    )
    results.append(metrics)
    trained_models[model_name] = fitted
    print(f"Acc={metrics['Accuracy']:.4f}  F1={metrics['F1-Score']:.4f}  "
          f"AUC={metrics['ROC-AUC']:.4f}  Time={metrics['Train Time (s)']}s ✅")

print('\nAll models trained and evaluated! 🎉')

---
## 6. Confusion Matrices

In [ ]:
n_models = len(trained_models)
fig, axes = plt.subplots(1, n_models, figsize=(4.5 * n_models, 4.5))

cmaps = ['Blues', 'Oranges', 'Greens', 'Purples', 'Reds']

for ax, (name, fitted_pipe), cmap in zip(axes, trained_models.items(), cmaps):
    y_pred = fitted_pipe.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['No Disease', 'Disease'])
    disp.plot(ax=ax, colorbar=False, cmap=cmap)
    ax.set_title(name, fontweight='bold', fontsize=10)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    # Rotate tick labels for readability
    ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')

plt.suptitle('Confusion Matrices — All Models (Test Set)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Cross-Validation Score Distribution

In [ ]:
# Re-compute CV scores for box-plot (using stored pipelines re-fitted above)
cv_data = {}
for name, fitted_pipe in trained_models.items():
    scores = cross_val_score(fitted_pipe, X_train, y_train, cv=skf, scoring='f1', n_jobs=-1)
    cv_data[name] = scores

fig, ax = plt.subplots(figsize=(10, 5))
box = ax.boxplot(
    list(cv_data.values()),
    labels=list(cv_data.keys()),
    patch_artist=True,
    notch=False,
    widths=0.4,
    medianprops=dict(color='black', linewidth=2)
)

colors = ['#5b9bd5', '#ed7d31', '#70ad47', '#a020f0', '#ff0000']
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title(f'{CV_FOLDS}-Fold Cross-Validation F1 Score Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('F1 Score')
ax.set_xlabel('Model')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
ax.set_xticklabels(list(cv_data.keys()), rotation=15, ha='right')
plt.tight_layout()
plt.show()

---
## 8. Results Summary Table

In [ ]:
def build_summary_table(results_list, rank_by='F1-Score'):
    """Convert results list to a styled, ranked DataFrame."""
    df_res = pd.DataFrame(results_list)
    df_res = df_res.sort_values(rank_by, ascending=False).reset_index(drop=True)
    df_res.index = df_res.index + 1          # rank starts at 1
    df_res.index.name = 'Rank'

    display_cols = [
        'Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score',
        'ROC-AUC', 'CV Summary', 'Train Time (s)'
    ]
    return df_res[display_cols]


summary = build_summary_table(results, rank_by='F1-Score')

# Pretty display
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
styled = (
    summary.style
    .background_gradient(subset=metric_cols, cmap='YlGn', vmin=0.7, vmax=1.0)
    .background_gradient(subset=['Train Time (s)'], cmap='YlOrRd_r')
    .format({col: '{:.4f}' for col in metric_cols})
    .format({'Train Time (s)': '{:.3f}'})
    .set_caption('📊 Model Comparison — Ranked by F1-Score (descending)')
)
styled

In [ ]:
# Plain-text version (for quick reference)
print(summary.to_string())

---
## 9. Performance Visualisations

In [ ]:
# ── 9a. Side-by-side bar chart for all metrics ────────────────────────────────
metric_cols_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
model_names = summary['Model'].tolist()

x = np.arange(len(model_names))
width = 0.14
palette = ['#5b9bd5', '#ed7d31', '#70ad47', '#a020f0', '#e74c3c']

fig, ax = plt.subplots(figsize=(14, 6))

for i, (metric, color) in enumerate(zip(metric_cols_plot, palette)):
    vals = summary[metric].values
    offset = (i - len(metric_cols_plot) / 2 + 0.5) * width
    bars = ax.bar(x + offset, vals, width, label=metric, color=color, alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=10, ha='right')
ax.set_ylim(0.0, 1.15)
ax.set_ylabel('Score')
ax.set_title('All Evaluation Metrics — Side-by-Side Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', ncol=3)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.axhline(y=0.8, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax.text(len(model_names) - 0.5, 0.81, '0.80 reference', fontsize=8, color='gray')

plt.tight_layout()
plt.show()

In [ ]:
# ── 9b. Individual metric bar charts (one subplot per metric) ─────────────────
fig, axes = plt.subplots(1, len(metric_cols_plot), figsize=(16, 5), sharey=False)

for ax, metric, color in zip(axes, metric_cols_plot, palette):
    vals = summary[metric].values
    bars = ax.bar(model_names, vals, color=color, alpha=0.85, edgecolor='white', width=0.55)
    ax.set_title(metric, fontweight='bold')
    ax.set_ylim(max(0, vals.min() - 0.05), min(1.05, vals.max() + 0.07))
    ax.set_xticklabels(model_names, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Score')
    # Annotate bars
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

plt.suptitle('Metric-wise Model Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 9c. Training time bar chart ───────────────────────────────────────────────
times = summary['Train Time (s)'].values
names = summary['Model'].tolist()

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(names, times, color='#ffc000', edgecolor='white', height=0.5)
ax.set_xlabel('Training Time (seconds)')
ax.set_title('Model Training Time Comparison', fontsize=14, fontweight='bold')
ax.invert_yaxis()

for bar, val in zip(bars, times):
    ax.text(bar.get_width() + max(times) * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}s', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── 9d. Radar chart — multi-metric overview ───────────────────────────────────
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

radar_metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
N = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
radar_colors = ['#5b9bd5', '#ed7d31', '#70ad47', '#a020f0', '#e74c3c']

for (_, row), color in zip(summary.iterrows(), radar_colors):
    values = row[radar_metrics].values.flatten().tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=row['Model'], color=color)
    ax.fill(angles, values, alpha=0.08, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_metrics, fontsize=11)
ax.set_ylim(0.5, 1.05)
ax.set_title('Radar Chart — Multi-Metric Model Overview',
             fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── 9e. Cross-validation mean F1 with error bars ──────────────────────────────
cv_means = [r['CV Mean F1'] for r in results]
cv_stds  = [r['CV Std F1']  for r in results]
model_labels = [r['Model'] for r in results]

# Sort by CV mean
sort_idx = np.argsort(cv_means)[::-1]
cv_means  = [cv_means[i]  for i in sort_idx]
cv_stds   = [cv_stds[i]   for i in sort_idx]
model_labels = [model_labels[i] for i in sort_idx]

fig, ax = plt.subplots(figsize=(9, 5))
x_pos = np.arange(len(model_labels))
bars = ax.bar(x_pos, cv_means, yerr=cv_stds, capsize=5,
              color='#2e86ab', alpha=0.8, edgecolor='white', width=0.5)

ax.set_xticks(x_pos)
ax.set_xticklabels(model_labels, rotation=15, ha='right')
ax.set_ylabel('Mean F1 Score')
ax.set_title(f'{CV_FOLDS}-Fold CV Mean F1 ± Std (Training Data)', fontsize=13, fontweight='bold')
ax.set_ylim(max(0, min(cv_means) - 0.1), min(1.05, max(cv_means) + 0.1))

for bar, mean, std in zip(bars, cv_means, cv_stds):
    ax.text(bar.get_x() + bar.get_width() / 2, mean + std + 0.005,
            f'{mean:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 10. Final Recommendation

In [ ]:
# ── Auto-generate recommendation ──────────────────────────────────────────────
best_row   = summary.iloc[0]   # sorted by F1-Score
second_row = summary.iloc[1] if len(summary) > 1 else None

best_name  = best_row['Model']
best_f1    = best_row['F1-Score']
best_auc   = best_row['ROC-AUC']
best_acc   = best_row['Accuracy']
best_rec   = best_row['Recall']
best_time  = best_row['Train Time (s)']
best_cv    = best_row['CV Summary']

# Determine train-time category
all_times = summary['Train Time (s)'].values
time_rank  = np.argsort(all_times)
best_time_rank = list(summary['Model']).index(best_name)
fastest_model  = summary.iloc[np.argmin(all_times)]['Model']

print('=' * 72)
print('  📋  FINAL MODEL RECOMMENDATION')
print('=' * 72)
print()
print(f'  Best model  : {best_name}')
print(f'  F1-Score    : {best_f1:.4f}')
print(f'  ROC-AUC     : {best_auc:.4f}')
print(f'  Accuracy    : {best_acc:.4f}')
print(f'  Recall      : {best_rec:.4f}')
print(f'  5-Fold CV   : {best_cv}  (F1, on training data)')
print(f'  Train time  : {best_time:.3f} s')
print()
print(
    f"  ✅  Best model is '{best_name}' because it achieves the highest F1-score "
    f"({best_f1:.4f}) on the held-out test set, indicating strong balance between "
    f"Precision and Recall — both critical for medical diagnosis where false "
    f"negatives carry significant risk. It also delivers a strong ROC-AUC of "
    f"{best_auc:.4f} (excellent discriminative power), a high Recall of {best_rec:.4f} "
    f"(correctly identifies {best_rec*100:.1f}% of actual heart disease cases), and "
    f"robust cross-validation performance ({best_cv}), confirming generalisation "
    f"rather than overfitting to the test set."
)
print()
if second_row is not None:
    print(
        f"  🥈  Runner-up is '{second_row['Model']}' "
        f"(F1={second_row['F1-Score']:.4f}, AUC={second_row['ROC-AUC']:.4f}). "
        f"It is a strong alternative, especially if training time is a priority "
        f"(trained in {second_row['Train Time (s)']:.3f}s)."
    )
print()
print(
    f"  💡  Note: In clinical settings, maximising Recall (sensitivity) is usually "
    f"preferred over maximising Precision to minimise missed diagnoses.  "
    f"Consider tuning decision thresholds or using class-weight='balanced' "
    f"if the deployment data is more imbalanced."
)
print()
print('=' * 72)

---
## Appendix: Full Numeric Summary

In [ ]:
# Print the full summary table one more time for easy copy-paste
full_summary = pd.DataFrame(results).sort_values('F1-Score', ascending=False).reset_index(drop=True)
full_summary.index += 1
full_summary.index.name = 'Rank'
display_cols = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score',
                'ROC-AUC', 'CV Summary', 'Train Time (s)']
print(full_summary[display_cols].to_string())